# GeoSIMCP 西雅图住房价格严格复现实验

## 实验目的

本实验使用西雅图住房价格数据复现 GeoSIMCP 方法，并对作者公开代码的数据划分流程进行规范化。

原始实验代码采用训练集、验证集和测试集三部分。为了避免测试集参与超参数选择，本实验采用以下四部分划分：

- **训练集（Train，70%）**：训练 XGBoost 点预测模型；
- **校准集（Calibration，10%）**：计算 conformal nonconformity scores；
- **验证集（Validation，10%）**：选择 GeoSIMCP 的 `bandwidth` 和 `lambda`；
- **测试集（Test，10%）**：在参数固定后进行一次最终评价。

实验保持作者原有的数据、预测目标、特征变量和 XGBoost 参数，仅规范数据划分及评估流程。

## 1. 实验环境与路径配置

本节完成以下工作：

1. 导入数据处理、机器学习和绘图所需的 Python 包；
2. 设置项目根目录、论文源码目录和数据文件路径；
3. 将作者提供的 GeoSIMCP 旧版源码加入 Python 搜索路径；
4. 导入 `GeoSIMCPzhao` 类。

该步骤只负责配置运行环境，不进行数据处理或模型训练。

In [3]:
# ============================================================
# 第1步：导入包并连接论文源码
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split


# 项目根目录
PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

# 论文旧版源码目录
MANUSCRIPT_CODE_DIR = (
    PROJECT_ROOT
    / "Codes_Manuscript_GeoSIMCP"
    / "Codes_GeoSIMCP"
    / "GeoConformal"
)

# 住房数据文件
DATA_PATH = (
    PROJECT_ROOT
    / "example"
    / "seattle_sample_3k.csv"
)


# 检查路径
if not MANUSCRIPT_CODE_DIR.exists():
    raise FileNotFoundError(
        f"找不到论文源码目录：{MANUSCRIPT_CODE_DIR}"
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"找不到住房数据：{DATA_PATH}"
    )


# 将论文源码加入 Python 搜索路径
code_dir_str = str(MANUSCRIPT_CODE_DIR)

if code_dir_str not in sys.path:
    sys.path.insert(0, code_dir_str)


# 导入 GeoSIMCP
from GeoSIMCPzhao import GeoSIMCPzhao


print("第1步完成")
print("论文源码目录：", MANUSCRIPT_CODE_DIR)
print("住房数据路径：", DATA_PATH)
print("XGBoost 版本：", xgb.__version__)
print("GeoSIMCP 类：", GeoSIMCPzhao)

第1步完成
论文源码目录： E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal
住房数据路径： E:\GeoSIMCP_Reproduce\geoconformal-main\example\seattle_sample_3k.csv
XGBoost 版本： 3.2.0
GeoSIMCP 类： <class 'GeoSIMCPzhao.GeoSIMCPzhao'>


## 2. 数据读取与变量构造

本节读取西雅图住房价格样本，并构造后续实验需要的目标变量、预测特征和空间坐标。

### 2.1 数据说明

原始数据包含 3000 条住房记录，主要变量包括：

- `bathrooms`：卫生间数量；
- `sqft_living`：住房面积；
- `sqft_lot`：土地面积；
- `grade`：房屋等级；
- `condition`：房屋状况；
- `waterfront`：是否临水；
- `view`：景观等级；
- `age`：房龄；
- `UTM_X`、`UTM_Y`：UTM 平面坐标；
- `log_price`：以 10 为底的房价对数。

作者将房价还原后除以 10000，因此本实验中的目标变量 `price` 以“万美元”为单位。

### 2.2 变量分工

- `X_aspatial`：8个非空间住房属性，用于计算 GeoSIMCP 的特征相似度；
- `X_spatial`：8个住房属性加 UTM 坐标，用于训练 XGBoost 预测模型；
- `y`：住房价格；
- `loc`：经纬度坐标，用于计算地理距离。

In [4]:
# ============================================================
# 第2步：读取数据并构造实验变量
# ============================================================

# 读取住房数据
data = pd.read_csv(DATA_PATH)

print("原始数据形状：", data.shape)
display(data.head())


# ------------------------------------------------------------
# 1. 将 UTM 坐标转换为经纬度
# ------------------------------------------------------------

data = gpd.GeoDataFrame(
    data,
    geometry=gpd.points_from_xy(
        data["UTM_X"],
        data["UTM_Y"]
    ),
    crs="EPSG:32610"
)

data = data.to_crs(epsg=4326)

# EPSG:4326 中 geometry.x 是经度，geometry.y 是纬度
data["longitude"] = data.geometry.x
data["latitude"] = data.geometry.y


# ------------------------------------------------------------
# 2. 将 log_price 还原为作者使用的 price
# 单位：万美元
# ------------------------------------------------------------

data["price"] = np.power(10, data["log_price"]) / 10000


# ------------------------------------------------------------
# 3. 定义特征、目标变量和空间坐标
# ------------------------------------------------------------

ASpatial_FEATURES = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age"
]

SPATIAL_FEATURES = ASpatial_FEATURES + [
    "UTM_X",
    "UTM_Y"
]

# 非空间属性：用于 GeoSIMCP 特征相似度
X_aspatial = data[ASpatial_FEATURES].copy()

# 空间预测特征：用于训练 XGBoost
X_spatial = data[SPATIAL_FEATURES].copy()

# 预测目标：房价，单位为万美元
y = data["price"].copy()

# 地理位置：经度、纬度
loc = data[["longitude", "latitude"]].copy()


# ------------------------------------------------------------
# 4. 检查数据
# ------------------------------------------------------------

print("\n第2步完成")
print("X_aspatial：", X_aspatial.shape)
print("X_spatial：", X_spatial.shape)
print("y：", y.shape)
print("loc：", loc.shape)

print("\n目标变量名称：", y.name)
print("房价范围：", y.min(), "至", y.max())
print("缺失值总数：", data.isna().sum().sum())
print("重复行数量：", data.duplicated().sum())

原始数据形状： (3000, 12)


,Unnamed: 0,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,UTM_X,UTM_Y,log_price
0,0,3.00,2660,4600,8,3,0,0,109,552217.557035,5.274945e+06,6.091315
1,1,2.25,2530,8736,7,4,0,0,57,565692.484331,5.272758e+06,5.790988
2,2,2.00,1390,13464,7,4,0,0,28,562451.661509,5.245291e+06,5.315130
3,3,1.00,940,4264,7,5,0,0,66,546816.935618,5.264407e+06,5.619093
4,4,2.25,2070,7225,8,3,0,0,36,564343.195352,5.244978e+06,5.477121



第2步完成
X_aspatial： (3000, 8)
X_spatial： (3000, 10)
y： (3000,)
loc： (3000, 2)

目标变量名称： price
房价范围： 8.2 至 363.5000000000002
缺失值总数： 0
重复行数量： 0


## 3. Train / Calibration / Validation / Test 四段数据划分

为了避免测试集参与模型训练或超参数选择，本实验将全部 3000 条住房记录划分为四部分：

- **训练集（Train，70%）**：2100 条，用于训练 XGBoost 点预测模型；
- **校准集（Calibration，10%）**：300 条，用于计算非一致性分数；
- **验证集（Validation，10%）**：300 条，用于选择 `bandwidth` 和 `lambda`；
- **测试集（Test，10%）**：300 条，在参数完全确定后进行最终评价。

所有特征、目标变量和空间坐标同时划分，以保证同一住房记录在各变量中的索引保持一致。随机种子设置为 42，以保证实验可以重复。

In [6]:
# ============================================================
# 第3步：四段数据划分
# 70% Train / 10% Calibration / 10% Validation / 10% Test
# ============================================================

# ------------------------------------------------------------
# 第一次划分：70% Train，30% Remaining
# ------------------------------------------------------------

(
    X_aspatial_train,
    X_aspatial_remaining,
    X_spatial_train,
    X_spatial_remaining,
    y_train,
    y_remaining,
    loc_train,
    loc_remaining
) = train_test_split(
    X_aspatial,
    X_spatial,
    y,
    loc,
    test_size=0.30,
    random_state=42
)


# ------------------------------------------------------------
# 第二次划分：从 Remaining 中取总数据的10%作为 Calibration
# Remaining 共900条，其中1/3为300条
# ------------------------------------------------------------

(
    X_aspatial_calib,
    X_aspatial_temp,
    X_spatial_calib,
    X_spatial_temp,
    y_calib,
    y_temp,
    loc_calib,
    loc_temp
) = train_test_split(
    X_aspatial_remaining,
    X_spatial_remaining,
    y_remaining,
    loc_remaining,
    train_size=1 / 3,
    random_state=42
)


# ------------------------------------------------------------
# 第三次划分：剩余600条平分为 Validation 和 Test
# ------------------------------------------------------------

(
    X_aspatial_validation,
    X_aspatial_test,
    X_spatial_validation,
    X_spatial_test,
    y_validation,
    y_test,
    loc_validation,
    loc_test
) = train_test_split(
    X_aspatial_temp,
    X_spatial_temp,
    y_temp,
    loc_temp,
    test_size=0.50,
    random_state=42
)


# ------------------------------------------------------------
# 汇总各数据集规模
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Dataset": [
        "Train",
        "Calibration",
        "Validation",
        "Test"
    ],
    "Sample size": [
        len(y_train),
        len(y_calib),
        len(y_validation),
        len(y_test)
    ],
    "Percentage": [
        len(y_train) / len(y) * 100,
        len(y_calib) / len(y) * 100,
        len(y_validation) / len(y) * 100,
        len(y_test) / len(y) * 100
    ]
})

display(split_summary)


# ------------------------------------------------------------
# 检查数据划分是否正确
# ------------------------------------------------------------

assert len(y_train) == 2100
assert len(y_calib) == 300
assert len(y_validation) == 300
assert len(y_test) == 300

assert set(y_train.index).isdisjoint(y_calib.index)
assert set(y_train.index).isdisjoint(y_validation.index)
assert set(y_train.index).isdisjoint(y_test.index)
assert set(y_calib.index).isdisjoint(y_validation.index)
assert set(y_calib.index).isdisjoint(y_test.index)
assert set(y_validation.index).isdisjoint(y_test.index)

assert (
    len(y_train)
    + len(y_calib)
    + len(y_validation)
    + len(y_test)
    == len(y)
)

print("第3步完成：四个数据集规模正确，且不存在样本重叠。")

,Dataset,Sample size,Percentage
0,Train,2100,70.0
1,Calibration,300,10.0
2,Validation,300,10.0
3,Test,300,10.0


第3步完成：四个数据集规模正确，且不存在样本重叠。


## 4. XGBoost 点预测模型训练与验证

GeoSIMCP 不是独立的点预测模型，而是在已有预测模型的基础上构造预测区间。本实验采用作者原代码中的 XGBoost 回归模型作为基础预测器。

### 4.1 模型输入与输出

- **输入变量**：8个住房属性以及 `UTM_X`、`UTM_Y` 两个空间坐标，共10个特征；
- **预测目标**：住房价格 `price`，单位为万美元；
- **训练数据**：仅使用训练集的2100条记录；
- **验证数据**：使用验证集检查点预测性能；
- **校准集**：本节不使用，留给后续 conformal calibration；
- **测试集**：本节不使用，留到所有参数确定后进行最终评价。

模型参数保持作者原始实验设置，不在本节进行额外调参。

In [7]:
# ============================================================
# 第4步：训练 XGBoost 点预测模型
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. 创建作者原参数的 XGBoost 回归模型
# ------------------------------------------------------------

model_spatial = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=3,
    min_child_weight=1.0,
    colsample_bytree=1.0
)


# ------------------------------------------------------------
# 2. 只使用 Train 训练点预测模型
# ------------------------------------------------------------

model_spatial.fit(
    X_spatial_train,
    y_train
)


# ------------------------------------------------------------
# 3. 在 Validation 上检查点预测性能
# 注意：此处没有使用 Test
# ------------------------------------------------------------

y_pred_validation = model_spatial.predict(
    X_spatial_validation
)

validation_mae = mean_absolute_error(
    y_validation,
    y_pred_validation
)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_validation
    )
)

validation_r2 = r2_score(
    y_validation,
    y_pred_validation
)


# ------------------------------------------------------------
# 4. 汇总结果
# ------------------------------------------------------------

model_performance = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R²"
    ],
    "Validation result": [
        validation_mae,
        validation_rmse,
        validation_r2
    ]
})

display(model_performance.round(4))


# ------------------------------------------------------------
# 5. 基本检查
# ------------------------------------------------------------

assert model_spatial.n_features_in_ == 10
assert len(y_pred_validation) == len(y_validation)
assert np.isfinite(y_pred_validation).all()

print("第4步完成：XGBoost 已仅使用训练集完成训练。")
print("训练样本数量：", len(y_train))
print("验证样本数量：", len(y_validation))
print("模型输入特征数：", model_spatial.n_features_in_)
print("测试集尚未使用：", len(y_test), "条")

,Metric,Validation result
0,MAE,7.7113
1,RMSE,11.7734
2,R²,0.8324


第4步完成：XGBoost 已仅使用训练集完成训练。
训练样本数量： 2100
验证样本数量： 300
模型输入特征数： 10
测试集尚未使用： 300 条


### 4.2 点预测结果说明

XGBoost 在验证集上的点预测结果如下：

- MAE 为 7.7113 万美元；
- RMSE 为 11.7734 万美元；
- R² 为 0.8324。

R² 表明模型能够解释约 83.24% 的住房价格差异，具备较好的点预测能力，可以作为后续 GeoSIMCP 预测区间的基础模型。

RMSE 高于 MAE，说明验证集中存在少量预测误差较大的住房样本。后续 GeoSIMCP 将根据校准残差、地理位置和住房属性相似性，为不同住房构造局部预测区间。

## 5. GeoSIMCP 校准与超参数选择

GeoSIMCP 根据校准集中的预测残差，为每个待预测住房构造局部预测区间。

局部权重同时考虑两类距离：

- **地理距离**：住房空间位置之间的距离；
- **属性距离**：住房面积、等级、房龄等非空间属性之间的差异。

两个关键超参数为：

- `bandwidth`：控制局部权重随联合距离衰减的速度；
- `lambda`：控制地理距离和属性距离的相对权重。

其中：

- `lambda = 1` 表示只考虑地理距离；
- `lambda = 0` 表示只考虑属性距离；
- `0 < lambda < 1` 表示同时使用两类距离。

本节使用 Calibration 集计算非一致性分数，并在 Validation 集上搜索最优参数。Test 集在本节中不参与任何计算。

In [8]:
# ============================================================
# 第5步：使用 Calibration 校准，在 Validation 上选择参数
# ============================================================

from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. 区间评分函数
# 数值越小，表示覆盖率和区间宽度的综合表现越好
# ------------------------------------------------------------

def interval_score(y_true, lower, upper, alpha=0.1):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower

    below_penalty = np.maximum(lower - y_true, 0)
    above_penalty = np.maximum(y_true - upper, 0)

    return (
        width
        + (2 / alpha) * below_penalty
        + (2 / alpha) * above_penalty
    )


# ------------------------------------------------------------
# 2. 保持作者原来的参数搜索范围
# ------------------------------------------------------------

bandwidth_values = np.linspace(0.1, 5.0, 20)
lambda_values = np.linspace(0.0, 1.0, 20)

alpha = 0.1
target_coverage = 1 - alpha

validation_records = []


# ------------------------------------------------------------
# 3. 网格搜索
# Calibration：计算校准残差
# Validation：评价每组参数
# Test：完全不使用
# ------------------------------------------------------------

for bandwidth in tqdm(
    bandwidth_values,
    desc="GeoSIMCP 参数搜索"
):
    for lambda_weight in lambda_values:

        geosimcp_validation = GeoSIMCPzhao(
            predict_f=model_spatial.predict,
            miscoverage_level=alpha,
            bandwidth=bandwidth,

            # 校准集空间坐标
            coord_calib=loc_calib.to_numpy(),

            # 验证集空间坐标
            coord_test=loc_validation.to_numpy(),

            # XGBoost 预测使用10个特征
            X_calib=X_spatial_calib,
            y_calib=y_calib,
            X_test=X_spatial_validation,
            y_test=y_validation,

            # 属性距离只使用8个非空间特征
            X_calib_weight=X_aspatial_calib,
            X_test_weight=X_aspatial_validation,

            lambda_weight=lambda_weight
        )

        result = geosimcp_validation.analyze()

        lower = np.asarray(result.lower_bound)
        upper = np.asarray(result.upper_bound)

        coverage = float(
            np.mean(
                (np.asarray(y_validation) >= lower)
                & (np.asarray(y_validation) <= upper)
            )
        )

        mean_width = float(
            np.mean(upper - lower)
        )

        mean_interval_score = float(
            np.mean(
                interval_score(
                    y_true=y_validation,
                    lower=lower,
                    upper=upper,
                    alpha=alpha
                )
            )
        )

        validation_records.append({
            "bandwidth": bandwidth,
            "lambda": lambda_weight,
            "coverage": coverage,
            "mean_width": mean_width,
            "interval_score": mean_interval_score
        })


# ------------------------------------------------------------
# 4. 整理参数搜索结果
# ------------------------------------------------------------

validation_results = pd.DataFrame(
    validation_records
)

eligible_results = validation_results[
    validation_results["coverage"] >= target_coverage
].copy()

if eligible_results.empty:
    raise RuntimeError(
        "当前参数范围内没有覆盖率达到0.90的组合。"
    )

best_validation_result = (
    eligible_results
    .sort_values(
        ["interval_score", "mean_width"],
        ascending=True
    )
    .iloc[0]
)


best_bandwidth = float(
    best_validation_result["bandwidth"]
)

best_lambda = float(
    best_validation_result["lambda"]
)


# ------------------------------------------------------------
# 5. 输出验证集最优参数
# ------------------------------------------------------------

print("\n第5步完成：验证集参数搜索结束。")
print(f"目标覆盖率：       {target_coverage:.3f}")
print(f"最佳 Bandwidth：   {best_bandwidth:.3f}")
print(f"最佳 Lambda：      {best_lambda:.3f}")
print(
    f"Validation Coverage："
    f"{best_validation_result['coverage']:.3f}"
)
print(
    f"Validation Mean Width："
    f"{best_validation_result['mean_width']:.3f}"
)
print(
    f"Validation Interval Score："
    f"{best_validation_result['interval_score']:.3f}"
)
print("Test 集尚未使用：", len(y_test), "条")


display(
    eligible_results
    .sort_values("interval_score")
    .head(10)
    .round(4)
)

D:\BaiduNetdiskDownload\Anaconda3\envs\geosimcp_repro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GeoSIMCP 参数搜索: 100%|██████████| 20/20 [00:09<00:00,  2.05it/s]


第5步完成：验证集参数搜索结束。
目标覆盖率：       0.900
最佳 Bandwidth：   0.100
最佳 Lambda：      1.000
Validation Coverage：0.913
Validation Mean Width：37.050
Validation Interval Score：53.384
Test 集尚未使用： 300 条


,bandwidth,lambda,coverage,mean_width,interval_score
19,0.1000,1.0,0.9133,37.0500,53.3841
39,0.3579,1.0,0.9233,37.9034,55.3243
79,0.8737,1.0,0.9233,37.4065,55.3992
99,1.1316,1.0,0.9233,37.4065,55.3992
139,1.6474,1.0,0.9233,37.4065,55.3992
119,1.3895,1.0,0.9233,37.4065,55.3992
159,1.9053,1.0,0.9233,37.4065,55.3992
179,2.1632,1.0,0.9233,37.4065,55.3992
259,3.1947,1.0,0.9233,37.4065,55.3992
199,2.4211,1.0,0.9233,37.4065,55.3992


### 5.1 验证集参数搜索结果

在覆盖率不低于目标值 0.90 的条件下，验证集选择出的最优参数为：

- `bandwidth = 0.100`；
- `lambda = 1.000`；
- Validation Coverage = 0.913；
- Validation Mean Width = 37.050 万美元；
- Validation Interval Score = 53.384。

`lambda = 1.000` 表明当前最优联合距离完全由地理距离决定，非空间住房属性距离的权重为0。因此，在本次数据划分下，最优 GeoSIMCP 实际上等价于只使用地理距离的 GeoCP。

同时，最优 `bandwidth` 位于当前搜索范围的下边界0.100。这可能说明更小的 bandwidth 值仍有进一步改善的可能，但为了保持当前实验流程的一致性，本阶段不扩大或修改参数搜索范围。

截至本节结束，测试集仍未参与模型训练、校准或超参数选择。

## 6. 独立测试集最终评价

本节固定验证集选择得到的最优参数：

- `bandwidth = 0.100`；
- `lambda = 1.000`。

随后使用 Calibration 集计算非一致性分数，并在此前完全未参与训练、校准和参数选择的 Test 集上构造预测区间。

本节报告以下指标：

- **Coverage**：测试样本真实值落入预测区间的比例；
- **Mean Width**：预测区间的平均完整宽度；
- **Interval Score**：同时考虑区间宽度和未覆盖惩罚的综合指标；
- **MAE、RMSE 和 R²**：XGBoost 点预测模型在测试集上的性能。

该结果才是本实验的独立测试集最终结果。

In [9]:
# ============================================================
# 第6步：固定参数，在独立 Test 集上评价一次
# ============================================================

# ------------------------------------------------------------
# 1. 测试集点预测结果
# ------------------------------------------------------------

y_pred_test = model_spatial.predict(
    X_spatial_test
)

test_mae = mean_absolute_error(
    y_test,
    y_pred_test
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_test
    )
)

test_r2 = r2_score(
    y_test,
    y_pred_test
)


# ------------------------------------------------------------
# 2. 使用固定参数构建最终 GeoSIMCP
# Calibration 用于校准，Test 只用于最终评价
# ------------------------------------------------------------

final_geosimcp = GeoSIMCPzhao(
    predict_f=model_spatial.predict,
    miscoverage_level=alpha,
    bandwidth=best_bandwidth,

    coord_calib=loc_calib.to_numpy(),
    coord_test=loc_test.to_numpy(),

    X_calib=X_spatial_calib,
    y_calib=y_calib,

    X_test=X_spatial_test,
    y_test=y_test,

    X_calib_weight=X_aspatial_calib,
    X_test_weight=X_aspatial_test,

    lambda_weight=best_lambda
)

final_result = final_geosimcp.analyze()


# ------------------------------------------------------------
# 3. 提取预测区间
# ------------------------------------------------------------

test_lower = np.asarray(
    final_result.lower_bound
)

test_upper = np.asarray(
    final_result.upper_bound
)

y_test_array = np.asarray(
    y_test
)


# ------------------------------------------------------------
# 4. 计算测试集区间指标
# ------------------------------------------------------------

test_coverage = float(
    np.mean(
        (y_test_array >= test_lower)
        & (y_test_array <= test_upper)
    )
)

test_mean_width = float(
    np.mean(
        test_upper - test_lower
    )
)

test_interval_score = float(
    np.mean(
        interval_score(
            y_true=y_test_array,
            lower=test_lower,
            upper=test_upper,
            alpha=alpha
        )
    )
)


# ------------------------------------------------------------
# 5. 检查结果
# ------------------------------------------------------------

assert len(test_lower) == len(y_test)
assert len(test_upper) == len(y_test)
assert np.isfinite(test_lower).all()
assert np.isfinite(test_upper).all()
assert np.all(test_upper >= test_lower)


# ------------------------------------------------------------
# 6. 汇总最终结果
# ------------------------------------------------------------

final_test_results = pd.DataFrame({
    "Metric": [
        "Point MAE",
        "Point RMSE",
        "Point R²",
        "Coverage",
        "Mean full interval width",
        "Interval score",
        "Bandwidth",
        "Lambda"
    ],
    "Test result": [
        test_mae,
        test_rmse,
        test_r2,
        test_coverage,
        test_mean_width,
        test_interval_score,
        best_bandwidth,
        best_lambda
    ]
})

display(
    final_test_results.round(4)
)

print("第6步完成：独立测试集最终评价结束。")
print(f"Test Coverage：       {test_coverage:.3f}")
print(f"Test Mean Width：     {test_mean_width:.3f}")
print(f"Test Interval Score： {test_interval_score:.3f}")
print(f"固定 Bandwidth：      {best_bandwidth:.3f}")
print(f"固定 Lambda：         {best_lambda:.3f}")

,Metric,Test result
0,Point MAE,7.7072
1,Point RMSE,12.7672
2,Point R²,0.8571
3,Coverage,0.9300
4,Mean full interval width,36.6322
5,Interval score,58.7577
6,Bandwidth,0.1000
7,Lambda,1.0000


第6步完成：独立测试集最终评价结束。
Test Coverage：       0.930
Test Mean Width：     36.632
Test Interval Score： 58.758
固定 Bandwidth：      0.100
固定 Lambda：         1.000


### 6.1 独立测试集结果分析

固定验证集选择得到的参数后，模型在独立测试集上的结果为：

- Point MAE = 7.7072 万美元；
- Point RMSE = 12.7672 万美元；
- Point R² = 0.8571；
- Coverage = 0.930；
- Mean Full Interval Width = 36.6322 万美元；
- Interval Score = 58.7577。

测试集覆盖率为 0.930，高于预设的目标覆盖率 0.900，说明最终预测区间在独立测试集上达到了覆盖率要求。

平均完整区间宽度为 36.6322 万美元，即预测区间在点预测值两侧的平均半宽约为 18.3161 万美元。

测试集 Interval Score 高于验证集，说明模型在未参与参数选择的测试数据上出现了更大的区间误差或未覆盖惩罚。这属于独立测试评价中的正常现象，也说明不能使用验证集结果代替最终测试结果。

最优参数 `lambda = 1.000` 表明联合距离完全由地理距离决定，住房属性距离的权重为0。因此，本次严格实验中选择出的 GeoSIMCP 实际等价于 GeoCP。该结果说明，在当前数据划分、基础模型和参数搜索范围下，加入住房属性相似性并未进一步改善区间预测表现。

此外，最优 `bandwidth = 0.100` 位于搜索范围下边界，表明更小的 bandwidth 可能仍值得进行后续敏感性分析。但当前结果保留作者原来的参数范围，不在测试集结果基础上重新调参。

## 7. 实验总结

本实验对西雅图住房价格数据上的 GeoSIMCP 进行了严格复现，并采用 Train、Calibration、Validation 和 Test 四段数据划分：

- Train：训练 XGBoost 点预测模型；
- Calibration：计算 conformal nonconformity scores；
- Validation：选择 bandwidth 和 lambda；
- Test：在参数固定后进行一次最终评价。

最终独立测试集结果表明：

1. XGBoost 点预测模型具有较好的预测能力，测试集 R² 为 0.8571；
2. conformal 预测区间覆盖率为 0.930，达到了目标覆盖率0.900；
3. 平均完整区间宽度为36.6322万美元；
4. 最优 lambda 为1.000，说明当前最优模型仅依赖地理距离；
5. 在本次实验设置下，GeoSIMCP没有表现出相对于GeoCP的额外属性相似性增益；
6. 最优 bandwidth 位于搜索范围下边界，后续可通过敏感性分析研究更小的 bandwidth。

需要强调的是，本实验说明的是当前数据划分和模型设置下的结果，不能据此直接否定 GeoSIMCP 方法。是否具有稳定优势，还需要在相同数据划分下进一步比较 Vanilla CP、GeoCP 和 GeoSIMCP，并在多个随机种子或其他数据集上重复实验。

In [10]:
# ============================================================
# 第7步：检查哪些测试样本造成较高的 Interval Score
# ============================================================

test_width_each = test_upper - test_lower

test_outside_distance = (
    np.maximum(test_lower - y_test_array, 0)
    + np.maximum(y_test_array - test_upper, 0)
)

test_penalty_each = (2 / alpha) * test_outside_distance
test_score_each = test_width_each + test_penalty_each

score_diagnostics = pd.DataFrame({
    "true_price": y_test_array,
    "predicted_price": y_pred_test,
    "lower": test_lower,
    "upper": test_upper,
    "covered": test_outside_distance == 0,
    "outside_distance": test_outside_distance,
    "width": test_width_each,
    "penalty": test_penalty_each,
    "interval_score": test_score_each
}, index=y_test.index)

display(
    score_diagnostics
    .sort_values("interval_score", ascending=False)
    .head(10)
    .round(3)
)

print(
    "最高5个样本占总 Interval Score 的比例：",
    score_diagnostics["interval_score"]
    .nlargest(5)
    .sum()
    / score_diagnostics["interval_score"].sum()
)

,true_price,predicted_price,lower,upper,covered,outside_distance,width,penalty,interval_score
1036,254.600,170.003998,146.982,193.026,False,61.574,46.044,1231.484,1277.528
2528,60.000,136.130997,114.111,158.151,False,54.111,44.040,1082.219,1126.259
2825,159.800,230.695007,206.072,255.317,False,46.272,49.245,925.445,974.689
1374,95.000,143.255005,131.068,155.441,False,36.068,24.373,721.367,745.740
1830,227.115,180.313004,155.752,204.875,False,22.240,49.124,444.798,493.921
2503,70.600,101.425003,87.061,115.790,False,16.461,28.729,329.214,357.943
1898,125.000,153.863007,136.648,171.078,False,11.648,34.430,232.965,267.395
847,140.000,174.585007,150.023,199.147,False,10.023,49.124,200.465,249.589
776,58.005,88.540001,68.076,109.004,False,10.071,40.929,201.415,242.344
2988,35.000,64.373001,43.908,84.837,False,8.908,40.929,178.170,219.098


最高5个样本占总 Interval Score 的比例： 0.26198753383913254
